# TSP — Datos y métodos exactos

**Optimización DS2011 — fases 1 y 2.** Lectura de instancias TSPLIB, matriz de
distancias, costo de una ruta `L(π)`, y los dos métodos exactos (fuerza bruta
y Held-Karp) que sirven para validar en subconjuntos chicos.

In [1]:
import sys
from pathlib import Path

# ubicar la raiz del proyecto (src/tsp) sin importar desde donde se abra el notebook
RAIZ = Path.cwd()
if not (RAIZ / "src" / "tsp").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))
from tsp.datos import leer_tsp, matriz_distancias, longitud_ruta
from tsp.exactos import fuerza_bruta, held_karp

## 1. Leer una instancia TSPLIB

`berlin52`: 52 ciudades, óptimo conocido = 7542 (confirmado contra el archivo
`solutions` oficial de TSPLIB95).

In [2]:
inst = leer_tsp(RAIZ / "data" / "raw" / "berlin52.tsp")
print(f"instancia: {inst.nombre}  n={inst.dimension}")
inst.coords[:5]  # primeras 5 ciudades (x, y)

instancia: berlin52  n=52


array([[565., 575.],
       [ 25., 185.],
       [345., 750.],
       [945., 685.],
       [845., 655.]])

## 2. Matriz de distancias (d_ij) y costo de una ruta (L(π))

Redondeo TSPLIB: `int(sqrt(dx**2+dy**2)+0.5)`. `L(π)` incluye el regreso a la
ciudad de origen (es la función de costo, sección 5.2 del informe).

In [3]:
D = matriz_distancias(inst.coords)
print(f"D: matriz {D.shape}, simetrica={(D == D.T).all()}, diagonal en cero={(D.diagonal() == 0).all()}")

ruta_identidad = list(range(inst.dimension))  # 0,1,2,...,51 nada mas para probar
print(f"L(ruta identidad) = {longitud_ruta(ruta_identidad, D)}")

D: matriz (52, 52), simetrica=True, diagonal en cero=True
L(ruta identidad) = 22205


## 3. Métodos exactos: fuerza bruta vs. Held-Karp

Ambos garantizan el óptimo, pero fuerza bruta cuesta O(n!) y Held-Karp
O(n²·2ⁿ). Se comparan en subconjuntos de 8, 9 y 10 ciudades — deben coincidir
exactamente.

In [4]:
import time

filas = []
for k in (8, 9, 10):
    Dk = matriz_distancias(inst.coords[:k])

    t0 = time.perf_counter()
    _, costo_fb = fuerza_bruta(Dk)
    t_fb = time.perf_counter() - t0

    t0 = time.perf_counter()
    _, costo_hk = held_karp(Dk)
    t_hk = time.perf_counter() - t0

    assert costo_fb == costo_hk, f"no coinciden para k={k}"
    filas.append({"ciudades": k, "optimo": costo_fb, "tiempo_fuerza_bruta_s": round(t_fb, 4), "tiempo_held_karp_s": round(t_hk, 4)})

import pandas as pd
pd.DataFrame(filas)

,ciudades,optimo,tiempo_fuerza_bruta_s,tiempo_held_karp_s
0,8,2551,0.0084,0.0017
1,9,2820,0.0723,0.0040
2,10,2826,0.7711,0.0118


La tabla de arriba es justo la explosión combinatoria del informe (sección
2.2), pero con tiempos reales en vez de la tabla teórica: fuerza bruta casi se
multiplica por 10 en cada ciudad adicional; Held-Karp casi no se nota.

**Para cambiar algo en vivo:** el rango `(8, 9, 10)` de la celda anterior es
el único parámetro — se puede subir hasta ~12 ciudades con Held-Karp sin
problema; fuerza bruta ya no es práctica pasadas las 11-12.